## Concept 7 — Parent Document Retriever

### What is it?
There is a tension in chunk sizing:
- **Small chunks** → precise retrieval (query matches well) but LLM lacks surrounding context
- **Large chunks** → LLM has full context but query match is diluted by irrelevant text

Parent Document Retriever solves this by using **both**:
- Index **small chunks** (200 chars) for precise retrieval
- Store **large parent chunks** (2000 chars) separately
- When a small chunk matches, return its **parent** to the LLM

### Pipeline
```
Documents
  → Split into small child chunks (200 chars) → indexed in vector store
  → Split into large parent chunks (2000 chars) → stored separately

Query
  → Search small chunks (precise match)
  → Find their parent chunks
  → Return parents to LLM (full context)
  → LLM → Answer
```

### Limitation (what Concept 8 fixes)
Even parent chunks contain irrelevant sentences alongside the useful ones.
→ Concept 8 (Contextual Compression) strips irrelevant parts before passing to LLM.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import load_docs, get_embeddings, get_llm, get_prompt, format_docs, TEST_QUERIES, run_queries
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load raw documents
raw_docs   = load_docs()
embeddings = get_embeddings()
llm        = get_llm()
prompt     = get_prompt()

print(f"Loaded {len(raw_docs)} document(s)")

### Step 1 — Create Two Sets of Chunks
Small chunks for retrieval, large parent chunks to return to the LLM.

In [ ]:
# Small chunks — used for retrieval (precise match)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
child_chunks   = child_splitter.split_documents(raw_docs)

# Large parent chunks — returned to LLM (full context)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
parent_chunks   = parent_splitter.split_documents(raw_docs)

print(f"Child chunks (small, for retrieval):  {len(child_chunks)} chunks @ 200 chars")
print(f"Parent chunks (large, for LLM):       {len(parent_chunks)} chunks @ 2000 chars")

### Step 2 — Index Small Chunks in Vector Store
Only child chunks go into the vector store for retrieval.

In [ ]:
child_vectorstore = FAISS.from_documents(child_chunks, embeddings)
print("Child vectorstore ready")

### Step 3 — Find Parent for a Matched Child
When a small chunk matches, find which large parent chunk contains it.

In [ ]:
def find_parent(child_doc, parent_chunks):
    """Find the parent chunk that contains this child chunk's text."""
    child_text = child_doc.page_content
    for parent in parent_chunks:
        if child_text[:100] in parent.page_content:
            return parent
    return child_doc  # fallback: return child if no parent found

# See the size difference
query = TEST_QUERIES[1]
child_results = child_vectorstore.similarity_search(query, k=1)
parent_result = find_parent(child_results[0], parent_chunks)

print(f"Query: {query}")
print(f"\nChild chunk ({len(child_results[0].page_content)} chars):")
print(child_results[0].page_content)
print(f"\nParent chunk ({len(parent_result.page_content)} chars):")
print(parent_result.page_content[:500], "...")

### Step 4 — Full Parent Document Retriever Function

In [ ]:
def parent_doc_rag(query):
    # Step 1: search small chunks (precise match)
    child_results = child_vectorstore.similarity_search(query, k=4)

    # Step 2: get their parents (full context), deduplicate
    seen    = set()
    parents = []
    for child in child_results:
        parent = find_parent(child, parent_chunks)
        if parent.page_content not in seen:
            parents.append(parent)
            seen.add(parent.page_content)

    # Step 3: answer using parent chunks
    context  = format_docs(parents)
    response = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

### Test — Standard Queries

In [ ]:
run_queries(parent_doc_rag, label="Concept 7 — Parent Document Retriever")